In [1]:
%pip install pytest

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pytest
from pyspark.sql import SparkSession


def _create_spark_session():
    return SparkSession.builder \
    .appName('pytest') \
    .config("spark.jars", "/opt/spark/jars/iceberg-spark-runtime-3.5_2.12-1.6.0.jar") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .config("spark.sql.catalog.local.warehouse", "s3a://datalake/iceberg") \
    .getOrCreate()

    spark.sparkContext.setLogLevel("ERROR")

@pytest.fixture(scope="session")
def spark():
    return _create_spark_session()


def test_returned_orders(spark, target_table):
    """Test the percentage of returned orders"""

    src = spark.sql(f"""
        SELECT 
            ROUND(
                COUNT(DISTINCT CASE WHEN is_returned = 'true' THEN order_id END) / 
                COUNT(DISTINCT order_id),
            2) AS returned_percent
            
        FROM 
            {target_table}
    """).take(1)[0][0]

    expected = 0.02

    assert src == expected, f"❌ percentual de devolução {src} é diferente do esperado"   



In [3]:
import traceback
import inspect
import time
from datetime import datetime

def run_tests(test_funcs, fixture_funcs=None, target_table=None, log_path=None):
    """
    Runner estilo pytest com:
    - Injeção de fixtures
    - Saída formatada
    - Registro de logs em DataFrame Spark e opcionalmente salva como Iceberg
    """
    fixtures = {}
    logs = []

    # Carrega fixtures
    if fixture_funcs:
        if isinstance(fixture_funcs, dict):
            for name, fx in fixture_funcs.items():
                fixtures[name] = fx()
        elif isinstance(fixture_funcs, list):
            for fx in fixture_funcs:
                fixtures[fx.__name__] = fx()

    # Adiciona o argumento `target_table` no escopo
    if target_table:
        fixtures['target_table'] = target_table

    for func in test_funcs:
        start = time.time()
        test_name = func.__name__
        desc = (func.__doc__ or "").strip()
        try:
            sig = inspect.signature(func)
            kwargs = {
                pname: fixtures[pname]
                for pname in sig.parameters
                if pname in fixtures
            }

            func(**kwargs)

            duration = time.time() - start
            print(f"✅ {test_name} PASSED ({duration:.2f}s)")
            if desc:
               print(f"📘 {desc}")
            logs.append((test_name, "PASSED", "", duration, datetime.now()))

        except AssertionError as e:
            duration = time.time() - start
            print(f"❌ {test_name} FAILED: {e} ({duration:.2f}s)")
            logs.append((test_name, "FAILED", str(e), duration, datetime.now()))

        except Exception as e:
            duration = time.time() - start
            print(f"💥 {test_name} ERROR: ({duration:.2f}s)")
            traceback.print_exc()
            logs.append((test_name, "ERROR", str(e), duration, datetime.now()))

    # capturar os logs dos testes
    spark = fixtures.get("spark")
    if spark and logs:
        log_df = spark.createDataFrame(logs, ["test_name", "status", "message", "duration", "timestamp"]) 
        

        if log_path:
            log_df.write.mode("append").format("iceberg").save(log_path)
            print(f"\n📁 Logs salvos em: {log_path}")
        else:
            print("\n📋 Resultado dos testes:")
            log_df.show()


In [4]:
 target_table ='iceberg.bronze.tbl_bronze_order_events'

run_tests(
    test_funcs=[test_returned_orders],
    fixture_funcs={'spark': _create_spark_session},
    target_table=target_table,
    log_path=None  # None to show in notbooks, path to write in iceberg format
)


25/11/16 15:57:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/16 15:57:23 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/11/16 15:57:23 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/11/16 15:57:23 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/11/16 15:57:23 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/11/16 15:57:23 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
25/11/16 15:57:23 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
25/11/16 15:57:23 WARN Utils: Service 'SparkUI' could not bind on port 4046. Attempting port 4047.
SLF4J: Failed to load class "org.s

✅ test_returned_orders PASSED (16.80s)
📘 Test the percentage of returned orders

📋 Resultado dos testes:


+--------------------+------+-------+------------------+--------------------+
|           test_name|status|message|          duration|           timestamp|
+--------------------+------+-------+------------------+--------------------+
|test_returned_orders|PASSED|       |16.799511909484863|2025-11-16 15:57:...|
+--------------------+------+-------+------------------+--------------------+

